# Predicting Geomagnetic Storms with Flarient

*Reproducible research notebook from [Flarient](https://flarient.com) — the space weather intelligence platform.*

**About this notebook:** This notebook is part of the [Flarient Research Notebooks](https://github.com/flarientglobal/flarient-notebooks) collection. It uses public data from NOAA SWPC, NASA, and the Flarient API.


## 1. Introduction

Geomagnetic storms are disturbances in Earth's magnetosphere caused by solar wind. In this notebook, we'll explore how to predict them using publicly available data from NOAA SWPC and the Flarient platform.


In [ ]:
# Import libraries
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Set plotting style
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Fetching Kp Index Data

The Kp index is a global measure of geomagnetic activity. Values of 5+ indicate geomagnetic storms (G1-G5 scale).


In [ ]:
# Fetch Kp data from NOAA SWPC
kp_url = "https://services.swpc.noaa.gov/json/planetary_k_index_1m.json"
response = requests.get(kp_url, timeout=30)
kp_data = response.json()

# Convert to DataFrame
df = pd.DataFrame(kp_data)
df['time_tag'] = pd.to_datetime(df['time_tag'])
df['kp'] = df['kp'].astype(float)
df = df.set_index('time_tag')

print(f"Fetched {len(df)} Kp readings")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()


## 3. Visualising Kp Over Time

Let's plot the Kp index to identify storm periods.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['kp'], color='#22d3ee', linewidth=1)
ax.axhline(y=5, color='#f59e0b', linestyle='--', label='G1 storm threshold (Kp=5)')
ax.axhline(y=6, color='#f97316', linestyle='--', label='G2 storm threshold (Kp=6)')
ax.axhline(y=7, color='#ef4444', linestyle='--', label='G3 storm threshold (Kp=7)')
ax.set_xlabel('Time')
ax.set_ylabel('Kp Index')
ax.set_title('Planetary K-Index (Last 30 Days) — Source: NOAA SWPC')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('kp_timeline.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Identifying Storm Events

Let's identify periods where Kp reached storm levels (Kp ≥ 5).


In [ ]:
# Find storm periods
storms = df[df['kp'] >= 5].copy()
storms['storm_start'] = storms.index
storms['gap'] = storms.index.to_series().diff()

# Group consecutive readings (gap < 3 hours = same storm)
storm_groups = (storms['gap'] > pd.Timedelta(hours=3)).cumsum()
storm_events = storms.groupby(storm_groups).agg(
    start=('kp', 'idxmin'),
    peak_kp=('kp', 'max'),
    duration_hours=('kp', 'count'),
)

print(f"Found {len(storm_events)} storm events:")
storm_events.head(10)


## 5. Fetching Flarient Forecasts

Now let's fetch forecasts from the Flarient API to compare with observed data.


In [ ]:
# Fetch Flarient event data
flarient_url = "https://flarient.com/api/functions/getSpaceEventsRssFeed"
try:
    response = requests.get(flarient_url, timeout=30)
    flarient_events = response.json()
    print(f"Fetched {len(flarient_events.get('events', []))} Flarient events")
except Exception as e:
    print(f"Flarient API: {e}")
    print("Visit https://flarient.com/space-events for live event data")


## 6. Comparing Predictions vs Outcomes

The key to forecasting credibility is comparing what was predicted before an event with what actually happened.


In [ ]:
# For each storm event, check if it was predicted
print("Storm Events vs Predictions:")
print("=" * 60)
for _, storm in storm_events.head(5).iterrows():
    peak_time = storm['start']
    peak_kp = storm['peak_kp']
    g_scale = f"G{int(peak_kp) - 2}" if peak_kp >= 3 else "Below threshold"
    print(f"  {peak_time}: Kp {peak_kp} ({g_scale}) — Duration: {storm['duration_hours']}h")

print("\nTo verify Flarient's predictions for these events, visit:")
print("  https://flarient.com/space-events")
print("  https://github.com/flarientglobal/flarient-event-ledger")


## 7. Building a Simple Prediction Model

Let's build a simple model using solar wind data to predict Kp.


In [ ]:
# Fetch solar wind data from NOAA
sw_url = "https://services.swpc.noaa.gov/products/ace/ace_swepam_1m.json"
sw_response = requests.get(sw_url, timeout=30)
sw_data = sw_response.json()

# Parse solar wind data (skip header row)
sw_df = pd.DataFrame(sw_data[1:], columns=sw_data[0])
sw_df['time_tag'] = pd.to_datetime(sw_df['time_tag'])
sw_df['speed'] = sw_df['speed'].astype(float)
sw_df['bz'] = sw_df['bz_gsm'].astype(float)
sw_df = sw_df.set_index('time_tag')

print(f"Solar wind data: {len(sw_df)} readings")
print(f"  Mean speed: {sw_df['speed'].mean():.0f} km/s")
print(f"  Min Bz: {sw_df['bz'].min():.1f} nT")

# Simple correlation: southward Bz + high speed → higher Kp
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(df.index, df['kp'], color='#22d3ee', label='Kp')
axes[0].set_ylabel('Kp Index')
axes[0].legend()
axes[1].plot(sw_df.index, sw_df['bz'], color='#f59e0b', label='Bz (nT)', alpha=0.7)
axes[1].axhline(y=0, color='white', linestyle='-', alpha=0.3)
axes[1].set_ylabel('Bz (nT)')
axes[1].legend()
plt.suptitle('Kp Index vs Solar Wind Bz — Source: NOAA SWPC')
plt.tight_layout()
plt.savefig('kp_vs_bz.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Conclusion

This notebook demonstrated:
1. Fetching real-time Kp data from NOAA
2. Identifying geomagnetic storm events
3. Comparing predictions with observed outcomes
4. Exploring the relationship between solar wind and geomagnetic activity

For more advanced forecasting tools and community predictions, visit [flarient.com](https://flarient.com).


---

## About Flarient

[Flarient](https://flarient.com) is a space weather intelligence platform providing real-time data, forecasts, and community-driven observations. Visit [flarient.com](https://flarient.com) for live space weather conditions, aurora forecasts, and more.

## License

MIT — This notebook is open source. [View on GitHub](https://github.com/flarientglobal/flarient-notebooks).
